In [ ]:
import random
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from transformers import AutoImageProcessor
from PIL import Image

from sklearn.model_selection import train_test_split

random.seed(19)
torch.manual_seed(19)

processor = AutoImageProcessor.from_pretrained(
    "google/vit-base-patch16-224"
)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=processor.image_mean,
        std=processor.image_std
    )
])

class CustomDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)  # 將資料夾路徑轉成Path物件，方便操作檔案。
        self.transform = transform

        self.image_paths = []
        self.labels = []
        valid_extensions = {".jpg"}

        # iterdir() 的功能是取得資料夾中的所有檔案及子資料夾路徑。
        # .suffix 取得副檔名，回傳.jpg
        # .stem取得不含附檔名的檔名
        # .lower()全部轉小寫
        # .startswitch("dog")判斷是否以dog開頭
        for image_path in sorted(self.root_dir.iterdir()):
            if image_path.suffix.lower() not in valid_extensions:
                continue
            filename = image_path.stem.lower()

            if filename.startswith("dog"):
                label = 0
            elif filename.startswith("cat"):
                label = 1
            else:
                continue

            self.labels.append(label)
            self.image_paths.append(image_path)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        label = self.labels[idx]

        with Image.open(image_path) as img:
            image = img.convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, label

dataset = CustomDataset(root_dir="../Data/train", transform=transform)

indices = list(range(len(dataset)))

train_indices, temp_indices = train_test_split(
    indices,
    test_size=0.2,
    random_state=6,
    stratify=dataset.labels
)

temp_labels = [dataset.labels[i] for i in temp_indices]

val_indices, test_indices = train_test_split(
    temp_indices,
    test_size=0.5,
    random_state=7,
    stratify=temp_labels
)

train_dataset = Subset(dataset, train_indices) # index對齊dataset
val_dataset = Subset(dataset, val_indices)
test_dataset = Subset(dataset, test_indices)


batch_size = 64
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)



c:\Users\Jian Yowen\anaconda3\envs\deep_hw2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch.nn as nn
from transformers import ViTModel
from torchvision.models import resnet50, ResNet50_Weights


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

criterion = nn.CrossEntropyLoss()

class ViTClassifier(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        # Load pretrained ViT-Base
        self.vit = ViTModel.from_pretrained(
            "google/vit-base-patch16-224",
            add_pooling_layer=False
        )

        # Freeze all ViT parameters
        for param in self.vit.parameters():
            param.requires_grad = False

        # ViT-Base hidden size = 768
        hidden_size = self.vit.config.hidden_size

        # New FC classification layer
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        outputs = self.vit(x)

        # 取出 CLS Token
        # 主要是為了從 ViT 的輸出中，取得代表整張圖片的特徵向量(Feature Vector)
        # 再交給最後的 FC Layer 進行貓狗分類。
        # : 取出batch中所有的圖片
        # 0 取出第0個token，也就是 CLS Token
        # :：取出該 Token 的全部 768 個特徵
        cls_token = outputs.last_hidden_state[:, 0, :]

        # FC Layer
        logits = self.fc(cls_token)

        return logits

vit_model = ViTClassifier(2).to(device)


class ResNet50Classifier(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        self.backbone = resnet50(
            weights=ResNet50_Weights.DEFAULT
        )

        in_features = self.backbone.fc.in_features

        self.backbone.fc = nn.Identity()

        # Freeze pretrained backbone
        for param in self.backbone.parameters():
            param.requires_grad = False

        # Trainable FC
        self.fc = nn.Linear(in_features, num_classes)

    def forward(self, x):
        features = self.backbone(x)
        logits = self.fc(features)
        return logits

resnet50_model = ResNet50Classifier(2).to(device)

vit_optimizer = torch.optim.Adam(
    vit_model.fc.parameters(),
    lr = 0.001,
)

resnet50_optimizer = torch.optim.Adam(
    resnet50_model.fc.parameters(),
    lr = 0.001,
)



Loading weights: 100%|██████████| 198/198 [00:00<00:00, 1590.56it/s]
[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def run_epoch(model, dataloader, criterion, optimizer=None):
    is_training = optimizer is not None

    if is_training:
        model.train()
    else:
        model.eval()

    total_loss = 0
    total_correct = 0
    total_samples = 0

    all_predictions = []
    all_labels = []
    all_probabilities = []

    for input_image, label in dataloader:
        input_image = input_image.to(device)
        label = label.to(device)

        with torch.set_grad_enabled(is_training):
            logit = model(input_image)
            loss = criterion(logit, label)

            if is_training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        predict = logit.argmax(dim=1)
        probabilities = torch.softmax(logit, dim=1)
        total_correct += (predict==label).sum().item()
        total_samples += len(label)
        total_loss += loss.item() * len(label)

        all_predictions.append(predict.detach().cpu())
        all_labels.append(label.detach().cpu())
        all_probabilities.append(probabilities.detach().cpu())

    all_predictions = torch.cat(all_predictions, dim=0).numpy()  # 因為每個batch的tensor都是一維，所以只能用dim=0
    all_labels = torch.cat(all_labels, dim=0).numpy()
    all_probabilities = torch.cat(all_probabilities, dim=0).numpy()

    accuracy = accuracy_score(all_labels, all_predictions)
    precision = precision_score(all_labels, all_predictions, average="macro", zero_division=0)
    recall = recall_score(all_labels, all_predictions, average="macro", zero_division=0)
    f1 = f1_score(all_labels, all_predictions, average="macro", zero_division=0)
    avg_loss = total_loss / total_samples

    macro_auc = roc_auc_score(all_labels, all_probabilities[:,1])
    

    return avg_loss, accuracy, precision, recall, f1, macro_auc

            

In [4]:
import copy
from tqdm.auto import tqdm

EPOCHS = 20
vit_best_val_loss = float("inf")
vit_best_model_state = None

progress_bar = tqdm(range(EPOCHS), desc="Training", unit="epoch")

for epoch in progress_bar:

    # Training
    train_loss, train_acc, train_precision, train_recall, train_f1, train_auc = run_epoch(
        vit_model, train_loader, criterion, vit_optimizer
    )

    # Validation
    val_loss, val_acc, val_precision, val_recall, val_f1, val_auc = run_epoch(
        vit_model, val_loader, criterion
    )

    # 更新最佳模型
    if val_loss < vit_best_val_loss:
        vit_best_val_loss = val_loss
        vit_best_model_state = copy.deepcopy(vit_model.state_dict())

    # 更新 tqdm 進度條
    progress_bar.set_postfix({
        "Train Loss": f"{train_loss:.4f}",
        "Val Loss": f"{val_loss:.4f}",
    })

    # 每個 Epoch 輸出完整 Metrics
    tqdm.write(
        f"Epoch [{epoch+1}/{EPOCHS}] | "
        f"Train: Loss={train_loss:.4f}, Acc={train_acc:.4f}, "
        f"Val: Loss={val_loss:.4f}, Acc={val_acc:.4f} "
    )

# 載入最佳模型
if vit_best_model_state is not None:
    vit_model.load_state_dict(vit_best_model_state)


Training:   5%|▌         | 1/20 [02:47<52:58, 167.26s/epoch, Train Loss=0.0249, Val Loss=0.0124]

Epoch [1/20] | Train: Loss=0.0249, Acc=0.9926, Val: Loss=0.0124, Acc=0.9960 


Training:  10%|█         | 2/20 [06:18<57:58, 193.26s/epoch, Train Loss=0.0125, Val Loss=0.0110]

Epoch [2/20] | Train: Loss=0.0125, Acc=0.9966, Val: Loss=0.0110, Acc=0.9968 


Training:  10%|█         | 2/20 [06:45<1:00:51, 202.85s/epoch, Train Loss=0.0125, Val Loss=0.0110]


KeyboardInterrupt: 

In [ ]:
import copy
from tqdm.auto import tqdm

EPOCHS = 20
resnet50_best_val_loss = float("inf")
resnet50_best_model_state = None

progress_bar = tqdm(range(EPOCHS), desc="Training", unit="epoch")

for epoch in progress_bar:

    # Training
    train_loss, train_acc, train_precision, train_recall, train_f1, train_auc = run_epoch(
        resnet50_model, train_loader, criterion, resnet50_optimizer
    )

    # Validation
    val_loss, val_acc, val_precision, val_recall, val_f1, val_auc = run_epoch(
        resnet50_model, val_loader, criterion
    )

    # 更新最佳模型
    if val_loss < resnet50_best_val_loss:
        resnet50_best_val_loss = val_loss
        resnet50_best_model_state = copy.deepcopy(resnet50_model.state_dict())

    # 更新 tqdm 進度條
    progress_bar.set_postfix({
        "Train Loss": f"{train_loss:.4f}",
        "Val Loss": f"{val_loss:.4f}",
    })

    # 每個 Epoch 輸出完整 Metrics
    tqdm.write(
        f"Epoch [{epoch+1}/{EPOCHS}] | "
        f"Train: Loss={train_loss:.4f}, Acc={train_acc:.4f}, "
        f"Val: Loss={val_loss:.4f}, Acc={val_acc:.4f} "
    )

# 載入最佳模型
if resnet50_best_model_state is not None:
    resnet50_model.load_state_dict(resnet50_best_model_state)


Training:   5%|▌         | 1/20 [01:17<24:36, 77.70s/epoch, Train Loss=0.0969, Val Loss=0.0403]

Epoch [1/20] | Train: Loss=0.0969, Acc=0.9729, Val: Loss=0.0403, Acc=0.9884 


Training:  10%|█         | 2/20 [02:35<23:17, 77.63s/epoch, Train Loss=0.0409, Val Loss=0.0347]

Epoch [2/20] | Train: Loss=0.0409, Acc=0.9875, Val: Loss=0.0347, Acc=0.9904 


Training:  15%|█▌        | 3/20 [03:52<21:56, 77.46s/epoch, Train Loss=0.0323, Val Loss=0.0313]

...

Epoch [18/20] | Train: Loss=0.0124, Acc=0.9963, Val: Loss=0.0319, Acc=0.9904 


Training:  95%|█████████▌| 19/20 [24:34<01:17, 77.71s/epoch, Train Loss=0.0118, Val Loss=0.0296]

Epoch [19/20] | Train: Loss=0.0118, Acc=0.9966, Val: Loss=0.0296, Acc=0.9912 


Training: 100%|██████████| 20/20 [25:52<00:00, 77.60s/epoch, Train Loss=0.0117, Val Loss=0.0315]

Epoch [20/20] | Train: Loss=0.0117, Acc=0.9973, Val: Loss=0.0315, Acc=0.9900 


In [ ]:

test_loss, test_acc, test_precision, test_recall, test_f1, test_auc = run_epoch(
    vit_model, test_loader, criterion
)

print("Vit test: ")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall: {test_recall:.4f}")
print(f"Test F1: {test_f1:.4f}")
print(f"Test auc: {test_auc:.4f}")

Vit test: 
Test Loss: 0.0126
Test Accuracy: 0.9956
Test Precision: 0.9956
Test Recall: 0.9956
Test F1: 0.9956
Test auc: 0.9999


In [ ]:

test_loss, test_acc, test_precision, test_recall, test_f1, test_auc = run_epoch(
    resnet50_model, test_loader, criterion
)

print("Resnet50 test: ")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall: {test_recall:.4f}")
print(f"Test F1: {test_f1:.4f}")
print(f"Test auc: {test_auc:.4f}")

Resnet50 test: 
Test Loss: 0.0256
Test Accuracy: 0.9896
Test Precision: 0.9896
Test Recall: 0.9896
Test F1: 0.9896
Test auc: 0.9996


In [ ]:
# Export the test set predictions of ResNet50 and ViT to CSV files.
import csv

class_names = {0: "dog", 1: "cat"}

# test_loader uses shuffle=False, so its order matches test_dataset.indices.
test_image_paths = [str(test_dataset.dataset.image_paths[idx]) for idx in test_dataset.indices]

def export_test_predictions(model, loader, image_paths, csv_filename):
    model.eval()
    rows = []
    image_offset = 0

    with torch.no_grad():
        for images, actual_labels in loader:
            predictions = model(images.to(device)).argmax(dim=1).cpu().tolist()
            actual_labels = actual_labels.tolist()

            for predicted, actual in zip(predictions, actual_labels):
                rows.append({
                    "image": image_paths[image_offset],
                    "predicted": class_names[predicted],
                    "actual": class_names[actual],
                })
                image_offset += 1

    assert image_offset == len(image_paths), "Test image count does not match predictions."
    with open(csv_filename, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=["image", "predicted", "actual"])
        writer.writeheader()
        writer.writerows(rows)

    print(f"Saved {len(rows)} test predictions to {csv_filename}")

export_test_predictions(resnet50_model, test_loader, test_image_paths, "resnet50_result.csv")
export_test_predictions(vit_model, test_loader, test_image_paths, "vit_result.csv")
